# Titanic exp009 - Add FareBand on top of Title + FamilyGroup + SexPclass

この notebook は現在のベスト構成 `Title + FamilyGroup + SexPclass` に `FareBand` を追加して比較するためのものです。

仮説:
- `Fare` は高いほど生存率が高いが、連続値としてよりも帯に区切った方が扱いやすいかもしれない
- `Title`、`FamilyGroup`、`SexPclass` では取り切れない料金帯の差を `FareBand` で補えるかもしれない

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    COMP_DIR = cwd.parent
elif cwd.name == "titanic":
    COMP_DIR = cwd
else:
    COMP_DIR = Path("/home/sora/dev/kaggle/competitions/titanic")

DATA_DIR = COMP_DIR / "data"
SUBMISSION_DIR = COMP_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
OUTPUT_PATH = SUBMISSION_DIR / "exp009_fareband_on_exp007_submission.csv"

FEATURE_COLUMNS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Title", "FamilyGroup", "SexPclass", "FareBand"]
NUMERIC_COLUMNS = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL_COLUMNS = ["Pclass", "Sex", "Embarked", "Title", "FamilyGroup", "SexPclass", "FareBand"]

TITLE_NORMALIZATION = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
    "Lady": "Rare",
    "Countess": "Rare",
    "Dona": "Rare",
    "Dr": "Rare",
    "Rev": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Capt": "Rare",
    "Sir": "Rare",
    "Don": "Rare",
    "Jonkheer": "Rare",
    "Master": "Master",
    "Miss": "Miss",
    "Mr": "Mr",
    "Mrs": "Mrs",
}


In [2]:
def add_features(df: pd.DataFrame, fare_bins=None) -> tuple[pd.DataFrame, pd.IntervalIndex]:
    out = df.copy()
    title = out["Name"].fillna("").astype(str).str.extract(r",\s*([^\.]+)\.", expand=False)
    out["Title"] = title.fillna("Rare").map(lambda x: TITLE_NORMALIZATION.get(x, "Rare")).astype(str)
    out["FamilySize"] = out["SibSp"].fillna(0) + out["Parch"].fillna(0) + 1
    out["FamilyGroup"] = pd.cut(
        out["FamilySize"],
        bins=[0, 1, 4, 100],
        labels=["alone", "small", "large"],
        right=True,
    ).astype(str)
    out["SexPclass"] = out["Sex"].fillna("missing").astype(str) + "_" + out["Pclass"].fillna(-1).astype(int).astype(str)

    fare_values = out["Fare"].fillna(out["Fare"].median())
    if fare_bins is None:
        fare_band = pd.qcut(fare_values, q=4, duplicates="drop")
        fare_bins = fare_band.cat.categories
    else:
        fare_band = pd.cut(fare_values, bins=fare_bins)

    out["FareBand"] = fare_band.astype(str)
    return out, fare_bins


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


In [3]:
train_df, fare_bins = add_features(pd.read_csv(TRAIN_PATH))
test_df, _ = add_features(pd.read_csv(TEST_PATH), fare_bins=fare_bins)

display(train_df[["Fare", "FareBand", "SexPclass", "Title", "FamilyGroup"]].head())
display(train_df.groupby("FareBand", observed=False)["Survived"].agg(["mean", "count"]).sort_index())
print("fare bins:", list(fare_bins))

,Fare,FareBand,SexPclass,Title,FamilyGroup
0,7.2500,"(-0.001, 7.91]",male_3,Mr,small
1,71.2833,"(31.0, 512.329]",female_1,Mrs,small
2,7.9250,"(7.91, 14.454]",female_3,Miss,alone
3,53.1000,"(31.0, 512.329]",female_1,Mrs,small
4,8.0500,"(7.91, 14.454]",male_3,Mr,alone


,mean,count
FareBand,,
"(-0.001, 7.91]",0.197309,223
"(14.454, 31.0]",0.454955,222
"(31.0, 512.329]",0.581081,222
"(7.91, 14.454]",0.303571,224


fare bins: [Interval(-0.001, 7.91, closed='right'), Interval(7.91, 14.454, closed='right'), Interval(14.454, 31.0, closed='right'), Interval(31.0, 512.329, closed='right')]


In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            NUMERIC_COLUMNS,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", make_one_hot_encoder()),
                ]
            ),
            CATEGORICAL_COLUMNS,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=3000,
                C=1.0,
                solver="lbfgs",
                random_state=42,
            ),
        ),
    ]
)


In [5]:
X = train_df[FEATURE_COLUMNS].copy()
y = train_df["Survived"].astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

print("CV scores:", [round(score, 5) for score in scores])
print("CV mean:", round(scores.mean(), 5))
print("CV std:", round(scores.std(), 5))

CV scores: [np.float64(0.85475), np.float64(0.82022), np.float64(0.82584), np.float64(0.82584), np.float64(0.83708)]
CV mean: 0.83275
CV std: 0.01229


比較メモ:

- `exp001` baseline CV mean: `0.79687`
- `exp006` Title + FamilyGroup CV mean: `0.82939`
- `exp007` Title + FamilyGroup + SexPclass CV mean: `0.83614`
- `exp009` exp007 + FareBand CV mean: 実行結果を記入

In [6]:
pipeline.fit(X, y)
test_predictions = pipeline.predict(test_df[FEATURE_COLUMNS].copy()).astype(int)

submission_df = pd.DataFrame(
    {
        "PassengerId": test_df["PassengerId"],
        "Survived": test_predictions,
    }
)
submission_df.to_csv(OUTPUT_PATH, index=False)

print("saved:", OUTPUT_PATH)
display(submission_df.head())

saved: /home/sora/dev/kaggle/competitions/titanic/submissions/exp009_fareband_on_exp007_submission.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
